# Differential Equations — Session 15
## Section 4.3: Homogeneous Constant-Coefficient Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Construct the characteristic equation.
2. Write solutions for distinct, repeated, and complex roots.
3. Solve IVPs and higher-order equations.
4. Interpret growth, decay, oscillation, and damping from roots.
5. Handle repeated complex roots.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | Exponential trial and characteristic polynomial |\n| 18–42 min | Three root cases |\n| 42–62 min | IVPs and root-based behavior |\n| 62–78 min | Higher-order/repeated roots |\n| 78–88 min | Interactive root explorer |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 4.3-A — Characteristic-root construction

For

$$
a_ny^{(n)}+\cdots+a_1y'+a_0y=0,
$$

substitution $y=e^{mx}$ gives the characteristic polynomial

$$
a_nm^n+\cdots+a_1m+a_0=0.
$$

### Root rules

- Distinct real root $m$: include $e^{mx}$.
- Real root $m$ of multiplicity $k$: include $e^{mx},xe^{mx},\ldots,x^{k-1}e^{mx}$.
- Complex pair $\alpha\pm i\beta$: include $e^{\alpha x}\cos\beta x$ and $e^{\alpha x}\sin\beta x$.
- Complex pair of multiplicity $k$: multiply both real solutions by $1,x,\ldots,x^{k-1}$.

### Behavioral interpretation

The real part controls exponential growth or decay; the imaginary part controls oscillation frequency.

### Classroom Checkpoint — Repeated Characteristic Root

If the characteristic polynomial has a repeated root $r$ of multiplicity two, what two independent solutions are used?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Three second-order cases

In [ ]:
def root_case(b=2.0,c=1.0,y0=1.0,v0=0.0,T=12):
    # y''+b y'+c y=0
    roots=np.roots([1,b,c]); print('roots:',roots)
    def rhs(t,z): return [z[1],-b*z[1]-c*z[0]]
    sol=solve_ivp(rhs,(0,T),[y0,v0],t_eval=np.linspace(0,T,900))
    plt.plot(sol.t,sol.y[0],linewidth=2); plt.axhline(0,linestyle='--'); plt.title(f'roots={roots[0]:.3g}, {roots[1]:.3g}'); plt.xlabel('t'); plt.ylabel('y'); plt.show()
if WIDGETS_AVAILABLE:
    interact(root_case,b=FloatSlider(min=-4,max=6,step=.25,value=2),c=FloatSlider(min=-4,max=10,step=.25,value=1),y0=FloatSlider(min=-3,max=3,step=.25,value=1),v0=FloatSlider(min=-5,max=5,step=.25,value=0),T=IntSlider(min=5,max=30,step=1,value=12))
else: root_case()

## 2. Example with complex roots

For

$$
4y''+4y'+17y=0,
$$

roots are $-1/2\pm2i$, so

$$
y=e^{-x/2}(c_1\cos2x+c_2\sin2x).
$$

In [ ]:
x=sp.symbols('x',real=True); c1,c2=sp.symbols('c1 c2')
y=sp.exp(-x/2)*(c1*sp.cos(2*x)+c2*sp.sin(2*x))
display(sp.simplify(4*sp.diff(y,x,2)+4*sp.diff(y,x)+17*y))

## 3. Root plane and qualitative behavior

In [ ]:
bs=np.linspace(-4,4,300); cs=np.linspace(-3,8,300); B,C=np.meshgrid(bs,cs); disc=B**2-4*C
plt.contourf(B,C,disc,levels=[-100,0,100],alpha=.7); plt.contour(B,C,disc,levels=[0],linewidths=2)
plt.xlabel('b'); plt.ylabel('c'); plt.title(r'Real versus complex roots for $m^2+bm+c=0$'); plt.show()

## 4. Higher-order multiplicity

If the characteristic polynomial is

$$
(m+1)^2(m^2+4)^2,
$$

then

$$
y=(c_1+c_2x)e^{-x}+(c_3+c_4x)\cos2x+(c_5+c_6x)\sin2x.
$$

In [ ]:
m=sp.symbols('m'); poly=sp.expand((m+1)**2*(m**2+4)**2); display(poly); display(sp.factor(poly))

## 5. Rational-root strategy

For higher-degree integer polynomials, rational candidates are $p/q$, with $p$ dividing the constant term and $q$ dividing the leading coefficient.

In [ ]:
m=sp.symbols('m'); p=3*m**3+5*m**2+10*m-4; display(sp.factor(p)); display(sp.solve(p,m))

## Classroom Checkpoint — Exit Check

Write the real general solution for roots $-2$ (multiplicity 2) and $1\pm3i$.

> Pause here. Let students commit to an answer before running the next cell.